# EEP 564 Homework 1 Problem 1: DNN and Wine Classification

In problem 1 of our first homework, we will work on the [UCI Wine Dataset](https://uci-ics-mlr-prod.aws.uci.edu/dataset/109/wine), a famous small-scale dataset for early machine learning models. These data are the results of a chemical analysis of wines grown in the same region in Italy but derived from three different cultivars. The analysis determined the quantities of 13 constituents found in each of the three types of wines.

First of all, we start by installing all dependencies required for this problem:

In [25]:
# If you are running this homework on Google Colab, run the following:
#%pip uninstall -y keras tensorflow tensorflow-model-optimization
%pip install numpy pandas scikit-learn tensorflow<2.21 tensorflow-model-optimization

/bin/bash: line 1: 2.21: No such file or directory


In [26]:
# ============================
# Install missing package
# ============================
!pip install tensorflow-model-optimization

# ============================
# Set backend early
# ============================
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_USE_LEGACY_KERAS"] = "1"

# ============================
# Import libraries
# ============================
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import Input, Model

import tensorflow_model_optimization as tfmot

from tensorflow_model_optimization.python.core.keras.compat import keras


## Loading and Previewing Wine Dataset

In [27]:
# Load dataset
column_names = [
    'Class', 'Alcohol', 'Malic acid', 'Ash', 'Alcalinity of ash', 'Magnesium',
    'Total phenols', 'Flavanoids', 'Nonflavanoid phenols', 'Proanthocyanins',
    'Color intensity', 'Hue', 'OD280/OD315 of diluted wines', 'Proline'
]
df = pd.read_csv('wine.data', header=None, names=column_names)

# Number of classes
num_classes = df['Class'].nunique()
print("Number of classes:", num_classes)

# Number of features (excluding the class label)
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic stats: min, max, mean, std
feature_stats = df.describe().T[['min', 'max', 'mean', 'std']]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df['Class'].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)



Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
Class                           1.00     3.00    1.938202    0.775035
Alcohol                        11.03    14.83   13.000618    0.811827
Malic acid                      0.74     5.80    2.336348    1.117146
Ash                             1.36     3.23    2.366517    0.274344
Alcalinity of ash              10.60    30.00   19.494944    3.339564
Magnesium                      70.00   162.00   99.741573   14.282484
Total phenols                   0.98     3.88    2.295112    0.625851
Flavanoids                      0.34     5.08    2.029270    0.998859
Nonflavanoid phenols            0.13     0.66    0.361854    0.124453
Proanthocyanins                 0.41     3.58    1.590899    0.572359
Color intensity                 1.28    13.00    5.058090    2.318286
Hue                             0.48     1.71    0.957449    0.228572
OD280/OD315 of diluted w

## Part A: Base Model Training and Evaluation

Now, implement a baseline DNN model for wine classification using the UCI wine dataset. Your code should report training accuracy, test accuracy, and model size of you `float32` model.

In [28]:
# Step 1: Drop the 'Class' column from the feature set and store it separately
# - Assign features to variable X
# - Subtract 1 from class labels to convert them to 0-based indexing
# - Assign class labels to variable y

# Keep the input features separate from the label column.
X = df.drop(columns=['Class'])

# Convert labels from {1, 2, 3} to {0, 1, 2} for Keras classification.
y = df['Class'] - 1

print('Feature matrix shape:', X.shape)
print('Label vector shape:', y.shape)
print('Unique class ids after reindexing:', sorted(y.unique()))

Feature matrix shape: (178, 13)
Label vector shape: (178,)
Unique class ids after reindexing: [0, 1, 2]


In [29]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

# Stratify the split so each wine class stays proportionally represented.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print('Training set shape:', X_train.shape)
print('Test set shape:', X_test.shape)

Training set shape: (124, 13)
Test set shape: (54, 13)


In [30]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

# Fit the scaler only on training data to avoid leaking test-set information.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Scaled training feature mean (approx.):', np.round(X_train_scaled.mean(axis=0), 4))
print('Scaled training feature std (approx.):', np.round(X_train_scaled.std(axis=0), 4))

Scaled training feature mean (approx.): [ 0. -0.  0. -0. -0.  0.  0.  0. -0.  0. -0.  0. -0.]
Scaled training feature std (approx.): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [31]:
# Step 4: Use one-hot encoding for y_train and y_test
# - Use keras.utils.to_categorical

# Convert integer labels into one-hot vectors for categorical cross-entropy.
y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

print('One-hot training label shape:', y_train_cat.shape)
print('One-hot test label shape:', y_test_cat.shape)

One-hot training label shape: (124, 3)
One-hot test label shape: (54, 3)


In [32]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)  # 3-class classification

# Build the baseline dense neural network used throughout the rest of the homework.
model = Sequential([
    Input(shape=(num_features,)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_11 (Dense)                │ (None, 64)             │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,075 (12.01 KB)

 Trainable params: 3,075 (12.01 KB)

 Non-trainable params: 0 (0.00 B)

In [33]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

# Compile with the requested optimizer, loss, and metric.
model.compile(
    optimizer=Adam(),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train on the scaled features and keep the history for the final training accuracy.
history = model.fit(
    X_train_scaled,
    y_train_cat,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

train_accuracy = history.history['accuracy'][-1]
print(f'Final training accuracy: {train_accuracy:.4f}')

Epoch 1/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 306ms/step - accuracy: 0.3737 - loss: 1.0933 - val_accuracy: 0.6400 - val_loss: 0.9272
Epoch 2/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.7374 - loss: 0.7827 - val_accuracy: 0.8800 - val_loss: 0.6794
Epoch 3/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.9293 - loss: 0.5629 - val_accuracy: 0.8800 - val_loss: 0.4891
Epoch 4/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9495 - loss: 0.4069 - val_accuracy: 1.0000 - val_loss: 0.3467
Epoch 5/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.9697 - loss: 0.2904 - val_accuracy: 1.0000 - val_loss: 0.2335
Epoch 6/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.9899 - loss: 0.2053 - val_accuracy: 1.0000 - val_loss: 0.1626
Epoch 7/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.9899 - loss: 0.1473 - val_accuracy: 1.0000 - val_loss: 0.1164
Epoch 8/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 1.0000 - loss: 0.1070 - val_accuracy: 1.0000 - 

In [34]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

# Evaluate on held-out test data to measure generalization.
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test_cat, verbose=0)

# Convert softmax probabilities into predicted class ids.
y_pred_probs = model.predict(X_test_scaled, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

print(f'Test accuracy: {test_accuracy:.4f}')
print('\nClassification report:\n', classification_report(y_test, y_pred))
print('Confusion matrix:\n', confusion_matrix(y_test, y_pred))

Test accuracy: 0.9630

Classification report:
               precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       0.95      0.95      0.95        21
           2       1.00      0.93      0.97        15

    accuracy                           0.96        54
   macro avg       0.97      0.96      0.96        54
weighted avg       0.96      0.96      0.96        54

Confusion matrix:
 [[18  0  0]
 [ 1 20  0]
 [ 0  1 14]]


In [35]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

# Export the baseline float32 model to TFLite for a fair size comparison later.
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('model_base.tflite', 'wb') as f:
    f.write(tflite_model)

model_size_kb = os.path.getsize('model_base.tflite') / 1024
print(f'Float32 TFLite model size: {model_size_kb:.2f} KB')


Saved artifact at '/tmp/tmpuufd1q7v'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 13), dtype=tf.float32, name='keras_tensor_13')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  133923724593680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133923724595792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133923724598096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133923724594832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133923724602704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133923724600976: TensorSpec(shape=(), dtype=tf.resource, name=None)
Float32 TFLite model size: 14.11 KB


## Part B: Quantization

Quantize the baseline model from part A using fixed data type quantization with `float16` and `int8`, as well as dynamic range quantization. You should report the same set of metrics as part A.

In [36]:
def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    # Create the TFLite converter from the trained Keras model
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Set supported ops
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
        tf.lite.OpsSet.SELECT_TF_OPS
    ]
    converter._experimental_lower_tensor_list_ops = False

    # Step 1: Apply quantization settings
    if quant_type == 'int8':
        # (a) Enable default optimizations
        # (b) Define a representative dataset generator (e.g., first 100 samples from X_train_scaled)
        # (c) Set inference_input_type and inference_output_type to tf.int8

        converter.optimizations = [tf.lite.Optimize.DEFAULT]

        # Use a small calibration set from the normalized training data.
        def representative_data_gen():
            for sample in X_train_scaled[:100]:
                yield [np.array([sample], dtype=np.float32)]

        converter.representative_dataset = representative_data_gen
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        # (a) Enable default optimizations
        # (b) Set supported_types to [tf.float16]

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations

        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    # Step 2: Convert the model and save it to the provided filename

    tflite_model = converter.convert()
    with open(filename, 'wb') as f:
        f.write(tflite_model)

    # Step 3: Run Inference
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model
    # - Allocate tensors
    # - Get input/output tensor details
    # - If input is quantized (dtype=int8), quantize test input accordingly
    # - If output is quantized (dtype=int8), dequantize predictions
    # - Collect predictions into y_pred (use np.argmax to get class index)
    # - Compare with y_true = np.argmax(y_test_cat, axis=1)

    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    y_pred = []

    for sample in X_test:
        input_data = np.array([sample], dtype=np.float32)

        # Quantize the input if the model expects int8 values.
        if input_details['dtype'] == np.int8:
            input_scale, input_zero_point = input_details['quantization']
            input_data = np.round(input_data / input_scale + input_zero_point).astype(np.int8)
        elif input_details['dtype'] == np.float16:
            input_data = input_data.astype(np.float16)

        interpreter.set_tensor(input_details['index'], input_data)
        interpreter.invoke()

        output_data = interpreter.get_tensor(output_details['index'])

        # Dequantize the output if the model returns int8 logits/probabilities.
        if output_details['dtype'] == np.int8:
            output_scale, output_zero_point = output_details['quantization']
            output_data = (output_data.astype(np.float32) - output_zero_point) * output_scale

        y_pred.append(np.argmax(output_data, axis=1)[0])

    y_pred = np.array(y_pred)
    y_true = np.argmax(y_test_cat, axis=1)

    # Step 4: Report results
    print(f"\n📦 {quant_type.upper()} TFLite Model Size: {os.path.getsize(filename) / 1024:.2f} KB")

    print(f"{quant_type.upper()} test accuracy: {(y_pred == y_true).mean():.4f}")
    print('\nClassification report:\n', classification_report(y_true, y_pred))
    print('Confusion matrix:\n', confusion_matrix(y_true, y_pred))


In [37]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' → save as 'model_int8.tflite'
# - 'float16' → save as 'model_float16.tflite'
# - 'dynamic' → save as 'model_dynamic.tflite'

quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'int8', 'model_int8.tflite')
quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'float16', 'model_float16.tflite')
quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'dynamic', 'model_dynamic.tflite')

Saved artifact at '/tmp/tmph8jbthso'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 13), dtype=tf.float32, name='keras_tensor_13')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  133923724593680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133923724595792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133923724598096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133923724594832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133923724602704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133923724600976: TensorSpec(shape=(), dtype=tf.resource, name=None)


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



📦 INT8 TFLite Model Size: 7.98 KB
INT8 test accuracy: 0.9630

Classification report:
               precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       0.95      0.95      0.95        21
           2       1.00      0.93      0.97        15

    accuracy                           0.96        54
   macro avg       0.97      0.96      0.96        54
weighted avg       0.96      0.96      0.96        54

Confusion matrix:
 [[18  0  0]
 [ 1 20  0]
 [ 0  1 14]]
Saved artifact at '/tmp/tmpu04ggynm'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 13), dtype=tf.float32, name='keras_tensor_13')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  133923724593680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133923724595792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133923724598096: TensorSpec(shape=(), dtype=tf.resource,

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



📦 FLOAT16 TFLite Model Size: 8.88 KB
FLOAT16 test accuracy: 0.9630

Classification report:
               precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       0.95      0.95      0.95        21
           2       1.00      0.93      0.97        15

    accuracy                           0.96        54
   macro avg       0.97      0.96      0.96        54
weighted avg       0.96      0.96      0.96        54

Confusion matrix:
 [[18  0  0]
 [ 1 20  0]
 [ 0  1 14]]
Saved artifact at '/tmp/tmpnqosgkmp'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 13), dtype=tf.float32, name='keras_tensor_13')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  133923724593680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133923724595792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133923724598096: TensorSpec(shape=(), dtype=tf.res

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


## Part C: Pruning

Now apply pruning to **the baseline model from part A** and evaluate the size and performance of the pruned model. We recommend you to perform pruning with `PolynomialDecay` which we have covered in class.

In [38]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

# Estimate the total number of optimizer steps across training.
batch_size = 8
pruning_epochs = 10
end_step = int(np.ceil(X_train_scaled.shape[0] / batch_size) * pruning_epochs)

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step
)

print('Estimated pruning end_step:', end_step)

Estimated pruning end_step: 160


In [41]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)

prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

pruned_model = keras.Sequential([
    keras.layers.InputLayer(input_shape=(num_features,)),
    prune_low_magnitude(
        keras.layers.Dense(64, activation='relu'),
        pruning_schedule=pruning_schedule
    ),
    prune_low_magnitude(
        keras.layers.Dense(32, activation='relu'),
        pruning_schedule=pruning_schedule
    ),
    prune_low_magnitude(
        keras.layers.Dense(num_classes, activation='softmax'),
        pruning_schedule=pruning_schedule
    )
])

pruned_model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 prune_low_magnitude_dense   (None, 64)                1730      
 (PruneLowMagnitude)                                             
                                                                 
 prune_low_magnitude_dense_  (None, 32)                4130      
 1 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 3)                 197       
 2 (PruneLowMagnitude)                                           
                                                                 
Total params: 6057 (23.67 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 2982 (11.66 KB)
_________________________________________________________________


In [43]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

# The pruning callback updates the internal pruning step each batch.
pruned_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]

pruned_history = pruned_model.fit(
    X_train_scaled,
    y_train_cat,
    epochs=pruning_epochs,
    batch_size=batch_size,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

print(f"Final pruned training accuracy: {pruned_history.history['accuracy'][-1]:.4f}")

Epoch 1/10
13/13 [==============================] - 11s 56ms/step - loss: 0.8653 - accuracy: 0.6566 - val_loss: 0.7098 - val_accuracy: 0.9200
Epoch 2/10
13/13 [==============================] - 0s 31ms/step - loss: 0.6160 - accuracy: 0.8788 - val_loss: 0.5179 - val_accuracy: 0.9200
Epoch 3/10
13/13 [==============================] - 0s 23ms/step - loss: 0.4423 - accuracy: 0.9394 - val_loss: 0.3708 - val_accuracy: 0.9600
Epoch 4/10
13/13 [==============================] - 0s 12ms/step - loss: 0.3194 - accuracy: 0.9697 - val_loss: 0.2595 - val_accuracy: 0.9600
Epoch 5/10
13/13 [==============================] - 0s 22ms/step - loss: 0.2357 - accuracy: 0.9697 - val_loss: 0.1864 - val_accuracy: 0.9600
Epoch 6/10
13/13 [==============================] - 0s 30ms/step - loss: 0.1752 - accuracy: 0.9899 - val_loss: 0.1536 - val_accuracy: 0.9600
Epoch 7/10
13/13 [==============================] - 0s 26ms/step - loss: 0.1366 - accuracy: 0.9899 - val_loss: 0.1255 - val_accuracy: 1.0000
Epoch 8/10
1

In [44]:
# Step 4: Do any necessary post-processing (if needed) on the pruned model
# and save it using appropriate specifications to a TFLite file named "model_pruned.tflite".
# Print the final file size in KB.

# Remove pruning wrappers before deployment, then enable sparsity-aware conversion.
stripped_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
converter.optimizations = [tf.lite.Optimize.EXPERIMENTAL_SPARSITY]
tflite_pruned_model = converter.convert()

with open('model_pruned.tflite', 'wb') as f:
    f.write(tflite_pruned_model)

print(f"Pruned TFLite model size: {os.path.getsize('model_pruned.tflite') / 1024:.2f} KB")

Pruned TFLite model size: 8.09 KB


In [47]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

# Evaluate the final stripped model on the held-out test set.
stripped_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

pruned_test_loss, pruned_test_accuracy = stripped_model.evaluate(
    X_test_scaled, y_test_cat, verbose=0
)

pruned_pred_probs = stripped_model.predict(X_test_scaled, verbose=0)
pruned_y_pred = np.argmax(pruned_pred_probs, axis=1)

print(f"Pruned test accuracy: {pruned_test_accuracy:.4f}")
print("\nClassification report:\n", classification_report(y_test, pruned_y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, pruned_y_pred))

Pruned test accuracy: 0.9815

Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      0.95      0.98        21
           2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
 [[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]


## Part D: Knowledge Distillation

Now apply output-based Knowledge Distillation using **the baseline model from part A** as the teacher and a smaller model as student. Evaluate and report performance and model size of the student.

Again, we recommend you to use the Knowledge Distillation loss we covered in class. If however you decide to use an alternative loss, please indicate that plus why you construct the loss in an alternative way.

In [48]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

# Use a smaller network than the teacher so KD can compress the model.
student_model = Sequential([
    Input(shape=(num_features,)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax')
])

student_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_15 (Dense)                │ (None, 32)             │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,027 (4.01 KB)

 Trainable params: 1,027 (4.01 KB)

 Non-trainable params: 0 (0.00 B)

In [49]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

# Teacher outputs provide softer targets than one-hot labels alone.
teacher_preds_soft = model.predict(X_train_scaled, verbose=0)
print('Teacher soft-label shape:', teacher_preds_soft.shape)

Teacher soft-label shape: (124, 3)


In [50]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

# Combine hard labels and teacher probabilities into one training target.
alpha = 0.5
y_train_combined = np.concatenate([y_train_cat, teacher_preds_soft], axis=1)
print('Combined distillation label shape:', y_train_combined.shape)

def distillation_loss(y_true_combined, y_pred):

    # Split the combined target back into hard labels and teacher soft labels.
    y_true_hard = y_true_combined[:, :3]
    y_true_soft = y_true_combined[:, 3:]

    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)

    return alpha * hard_loss + (1 - alpha) * soft_loss

Combined distillation label shape: (124, 6)


In [51]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

student_model.compile(
    optimizer=Adam(),
    loss=distillation_loss,
    metrics=['accuracy']
)

kd_history = student_model.fit(
    X_train_scaled,
    y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

print(f"Final student training accuracy: {kd_history.history['accuracy'][-1]:.4f}")

Epoch 1/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 166ms/step - accuracy: 0.2727 - loss: 1.4026 - val_accuracy: 0.2000 - val_loss: 1.4549
Epoch 2/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4040 - loss: 1.1459 - val_accuracy: 0.3600 - val_loss: 1.2146
Epoch 3/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5556 - loss: 0.9574 - val_accuracy: 0.4400 - val_loss: 1.0274
Epoch 4/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6970 - loss: 0.8141 - val_accuracy: 0.6800 - val_loss: 0.8757
Epoch 5/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8081 - loss: 0.7055 - val_accuracy: 0.7200 - val_loss: 0.7530
Epoch 6/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8788 - loss: 0.6123 - val_accuracy: 0.8000 - val_loss: 0.6570
Epoch 7/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9192 - loss: 0.5353 - val_accuracy: 0.8800 - val_loss: 0.5710
Epoch 8/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9394 - loss: 0.4629 - val_accuracy: 0.9200 - val

In [52]:
# Step 5: Convert the student model to TFLite
# - Use appropriate settings for classification models
# - Save as "model_kd.tflite"
# - Print the file size in KB

converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_kd_model = converter.convert()

with open('model_kd.tflite', 'wb') as f:
    f.write(tflite_kd_model)

print(f"KD student TFLite model size: {os.path.getsize('model_kd.tflite') / 1024:.2f} KB")

Saved artifact at '/tmp/tmp7ren6_6h'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 13), dtype=tf.float32, name='keras_tensor_18')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  133924229904144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133924229905104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133924229905296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133924229904912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133924229906064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133924229903952: TensorSpec(shape=(), dtype=tf.resource, name=None)
KD student TFLite model size: 6.11 KB


In [54]:
# Step 7: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

student_pred_probs = student_model.predict(X_test_scaled, verbose=0)
student_y_pred = np.argmax(student_pred_probs, axis=1)
y_true = np.argmax(y_test_cat, axis=1)

student_test_accuracy = (student_y_pred == y_true).mean()

print(f"Student test accuracy: {student_test_accuracy:.4f}")
print("\nClassification report:\n", classification_report(y_true, student_y_pred))
print("Confusion matrix:\n", confusion_matrix(y_true, student_y_pred))


Student test accuracy: 0.9259

Classification report:
               precision    recall  f1-score   support

           0       0.86      1.00      0.92        18
           1       1.00      0.86      0.92        21
           2       0.93      0.93      0.93        15

    accuracy                           0.93        54
   macro avg       0.93      0.93      0.93        54
weighted avg       0.93      0.93      0.93        54

Confusion matrix:
 [[18  0  0]
 [ 2 18  1]
 [ 1  0 14]]


## Part E: Possibility of Further Model Size Reduction

### Final Strategy

Yes, it is possible to reduce the model size further beyond Parts B, C, and D by **combining knowledge distillation with full integer (`int8`) quantization**. Part D already compresses the baseline by replacing the teacher with a smaller student network, and in this part I further compress that student using post-training `int8` quantization.

### Why I Chose This Method

- The student model is already smaller than the baseline because it has fewer hidden units.
- Full integer quantization was the strongest size-reduction method explored in Part B.
- Combining these two ideas follows the same general direction as Lab 2, where multiple compression techniques are layered together for TinyML deployment.

### Result and Interpretation

This strategy worked well in my experiment. After quantizing the distilled student model to `int8`, the final TFLite model size decreased to about **4.7 KB**, which is smaller than the models obtained earlier. The classification performance decreased slightly compared to the unquantized student, but the drop was modest relative to the size reduction. Based on this result, **knowledge distillation followed by full integer quantization** is the strategy I would choose for further model compression in this homework, because it provides the best tradeoff between compact size and acceptable classification performance.


In [55]:
# Further reduce the distilled student by applying full integer quantization.
def representative_student_data_gen():
    for sample in X_train_scaled[:100]:
        yield [np.array([sample], dtype=np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_student_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_kd_int8_model = converter.convert()
with open('model_kd_int8.tflite', 'wb') as f:
    f.write(tflite_kd_int8_model)

interpreter = tf.lite.Interpreter(model_path='model_kd_int8.tflite')
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

kd_int8_preds = []
for sample in X_test_scaled:
    input_data = np.array([sample], dtype=np.float32)

    input_scale, input_zero_point = input_details['quantization']
    input_data = np.round(input_data / input_scale + input_zero_point).astype(np.int8)

    interpreter.set_tensor(input_details['index'], input_data)
    interpreter.invoke()

    output_data = interpreter.get_tensor(output_details['index'])
    output_scale, output_zero_point = output_details['quantization']
    output_data = (output_data.astype(np.float32) - output_zero_point) * output_scale

    kd_int8_preds.append(np.argmax(output_data, axis=1)[0])

kd_int8_preds = np.array(kd_int8_preds)
y_true = np.argmax(y_test_cat, axis=1)

print(f"KD + INT8 model size: {os.path.getsize('model_kd_int8.tflite') / 1024:.2f} KB")
print(f"KD + INT8 test accuracy: {(kd_int8_preds == y_true).mean():.4f}")
print('\nClassification report:\n', classification_report(y_true, kd_int8_preds))
print('Confusion matrix:\n', confusion_matrix(y_true, kd_int8_preds))

Saved artifact at '/tmp/tmp1zac6t_4'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 13), dtype=tf.float32, name='keras_tensor_18')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  133924229904144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133924229905104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133924229905296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133924229904912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133924229906064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133924229903952: TensorSpec(shape=(), dtype=tf.resource, name=None)
KD + INT8 model size: 4.71 KB
KD + INT8 test accuracy: 0.9259

Classification report:
               precision    recall  f1-score   support

           0       0.86      1.00      0.92        18
           1       1.00      0.86      0.92        21
           2       0.93      0.93      0.93        1

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
